# 305.  SKT가 공개한 한글 Bert model 사용하기

https://github.com/monologg/KoBERT-Transformers

In [1]:
!pip install -q kobert_transformers

     |████████████████████████████████| 1.2 MB 8.5 MB/s 
     |████████████████████████████████| 2.8 MB 51.5 MB/s 
     |████████████████████████████████| 50 kB 6.3 MB/s 
     |████████████████████████████████| 895 kB 55.4 MB/s 
     |████████████████████████████████| 3.3 MB 35.2 MB/s 
     |████████████████████████████████| 636 kB 50.9 MB/s 


In [2]:
from kobert_transformers import get_tokenizer

tokenizer = get_tokenizer()
tokenizer.tokenize("한국어 모델을 공유합니다.")

Downloading:   0%|          | 0.00/371k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/77.8k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/426 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertTokenizer'. 
The class this function is called from is 'KoBertTokenizer'.


['▁한국', '어', '▁모델', '을', '▁공유', '합니다', '.']

In [3]:
tokenizer.convert_tokens_to_ids(['[CLS]', '▁한국', '어', '▁모델', '을', '▁공유', '합니다', '.', '[SEP]'])

[2, 4958, 6855, 2046, 7088, 1050, 7843, 54, 3]

In [4]:
pt_batch = tokenizer(
    ["한국어 모델을 공유합니다.", 
     "자연어 처리는 어렵지만 재미있고 보람도 있습니다."],
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)
pt_batch

{'input_ids': tensor([[   2, 4958, 6855, 2046, 7088, 1050, 7843,   54,    3,    1,    1,    1,
            1,    1,    1],
        [   2, 3916, 6855, 4465, 5760, 3231, 7330, 3978, 5439, 2355, 6018, 5859,
         3867,   54,    3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [5]:
for key, value in pt_batch.items():
    print(f"{key}: {value.numpy().tolist()}")

input_ids: [[2, 4958, 6855, 2046, 7088, 1050, 7843, 54, 3, 1, 1, 1, 1, 1, 1], [2, 3916, 6855, 4465, 5760, 3231, 7330, 3978, 5439, 2355, 6018, 5859, 3867, 54, 3]]
token_type_ids: [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
attention_mask: [[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]


In [6]:
from kobert_transformers import get_kobert_model, get_distilkobert_model
import torch

model = get_kobert_model()
model.eval()

input_ids      = torch.LongTensor(pt_batch.input_ids)
attention_mask = torch.LongTensor(pt_batch.attention_mask)
token_type_ids = torch.LongTensor(pt_batch.token_type_ids)

result = model(input_ids, attention_mask, token_type_ids)

Downloading:   0%|          | 0.00/369M [00:00<?, ?B/s]

In [7]:
result

BaseModelOutputWithPoolingAndCrossAttentions([('last_hidden_state',
                                               tensor([[[-1.5543e-01, -1.5011e-02,  3.6734e-01,  ..., -9.4244e-03,
                                                          1.0671e-01,  8.4437e-02],
                                                        [ 1.2287e-01, -3.2359e-01, -6.6921e-02,  ..., -4.4872e-01,
                                                         -1.7526e-01, -2.3015e-01],
                                                        [ 1.0082e-01, -3.8846e-01, -1.2187e-01,  ..., -2.1285e-01,
                                                         -3.2995e-02, -1.7076e-01],
                                                        ...,
                                                        [-1.5378e-01, -1.8186e-01, -1.9937e-01,  ..., -2.8618e-01,
                                                          1.3235e-02,  7.8320e-02],
                                                        [-1.5378e-01, -1.81

In [8]:
result['last_hidden_state'].shape

torch.Size([2, 15, 768])

In [9]:
result['pooler_output'].shape

torch.Size([2, 768])